# OmniSub 2026 — Lip Reading + LRS2 Matching

In [ ]:
# ── Cell 1: Install & clone auto_avsr ──
import os, subprocess, glob

!pip install -q jiwer sentencepiece scikit-image av
!pip install -q mediapipe==0.10.14
!pip install -q 'Pillow>=10.0'

import mediapipe as mp
print(f'mediapipe {mp.__version__}, solutions={hasattr(mp, "solutions")}')

AVSR_DIR = '/kaggle/working/auto_avsr'
if not os.path.exists(AVSR_DIR):
    !git clone --depth 1 https://github.com/mpc001/auto_avsr.git {AVSR_DIR}

MODEL_PATH = None
for c in glob.glob('/kaggle/input/**/vsr_model.pth', recursive=True):
    if os.path.getsize(c) > 1e6:
        MODEL_PATH = c
        break
if not MODEL_PATH:
    for c in glob.glob('/kaggle/input/**/*.pth', recursive=True):
        if os.path.getsize(c) > 1e8:
            MODEL_PATH = c
            break

print(f'Model: {MODEL_PATH} ({os.path.getsize(MODEL_PATH)/1e6:.0f} MB)' if MODEL_PATH else 'Model NOT FOUND')
print('Setup OK')

In [ ]:
# ── Cell 2: Discover paths & load LRS2 data ──
import os, csv, re, json, subprocess, time, sys, argparse
from pathlib import Path
from collections import defaultdict

def find_file(root, name, max_depth=4):
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath.replace(root, '').count(os.sep)
        if depth >= max_depth:
            dirnames.clear()
            continue
        if name in filenames or name in dirnames:
            return Path(dirpath) / name
    return None

SAMPLE_SUB = find_file('/kaggle/input', 'sample_submission.csv')
LRS2_DIR = find_file('/kaggle/input', 'lrs2_train_text.txt').parent
TEST_DIR = find_file('/kaggle/input', 'test')
TRAIN_DIR = find_file('/kaggle/input', 'train')
for d_attr in ['TEST_DIR', 'TRAIN_DIR']:
    d = eval(d_attr)
    if d:
        sub = d / d.name
        if sub.exists(): exec(f'{d_attr} = sub')
OUTPUT = Path('/kaggle/working/submission.csv')
print(f'TEST: {TEST_DIR}\nTRAIN: {TRAIN_DIR}\nLRS2: {LRS2_DIR}')

def norm(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

lrs2_by_channel = defaultdict(list)
for fname in ['lrs2_train_text.txt', 'lrs2_test_text.txt', 'lrs2_val_text.txt']:
    fpath = LRS2_DIR / fname
    if not fpath.exists(): continue
    with open(fpath) as f:
        for line in f:
            parts = line.strip().split(' ', 1)
            if len(parts) < 2: continue
            ch = parts[0].rsplit('_', 1)[0]
            lrs2_by_channel[ch].append(norm(parts[1]))
for ch in lrs2_by_channel:
    lrs2_by_channel[ch] = list(dict.fromkeys(lrs2_by_channel[ch]))
lrs2_all_texts = list(set(t for texts in lrs2_by_channel.values() for t in texts))
print(f'LRS2: {sum(len(v) for v in lrs2_by_channel.values())} texts, {len(lrs2_by_channel)} channels')

test_paths = []
with open(SAMPLE_SUB) as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader: test_paths.append(row[0])
paths_with_cand = [p for p in test_paths if p.split('/')[1] in lrs2_by_channel]
paths_no_cand = [p for p in test_paths if p.split('/')[1] not in lrs2_by_channel]
print(f'Test: {len(test_paths)} total, {len(paths_with_cand)} with cand, {len(paths_no_cand)} without')

In [ ]:
# ── Cell 3: Initialize auto_avsr lip reading pipeline (n-best) ──
import torch, torchvision, numpy as np

AVSR_DIR = '/kaggle/working/auto_avsr'
sys.path.insert(0, AVSR_DIR)

VSR_OK = False
pipeline = None

if MODEL_PATH and os.path.exists(MODEL_PATH):
    try:
        from lightning import ModelModule, get_beam_search_decoder
        from datamodule.transforms import VideoTransform, TextTransform
        from preparation.detectors.mediapipe.detector import LandmarksDetector
        from preparation.detectors.mediapipe.video_process import VideoProcess

        class VSRPipeline(torch.nn.Module):
            def __init__(self, model_path, device='cuda', n_best=10):
                super().__init__()
                self.device = device
                self.n_best = n_best
                self.landmarks_detector = LandmarksDetector()
                self.video_process = VideoProcess(convert_gray=False)
                self.video_transform = VideoTransform(subset='test')
                args = argparse.Namespace()
                args.modality = 'video'
                args.ctc_weight = 0.1
                ckpt = torch.load(model_path, map_location='cpu')
                self.modelmodule = ModelModule(args)
                self.modelmodule.model.load_state_dict(ckpt)
                self.modelmodule.eval()
                if device == 'cuda' and torch.cuda.is_available():
                    self.modelmodule = self.modelmodule.cuda()
                # Precompute beam search decoder once
                self.beam_search = get_beam_search_decoder(
                    self.modelmodule.model, self.modelmodule.token_list
                )
                self.text_transform = self.modelmodule.text_transform

            @torch.no_grad()
            def __call__(self, video_path):
                video_path = os.path.abspath(video_path)
                video = torchvision.io.read_video(video_path, pts_unit='sec')[0].numpy()
                landmarks = self.landmarks_detector(video)
                video = self.video_process(video, landmarks)
                if video is None: return ['']
                video = torch.tensor(video).permute(0, 3, 1, 2)
                video = self.video_transform(video)
                if self.device == 'cuda' and torch.cuda.is_available():
                    video = video.cuda()

                # Run encoder
                x = self.modelmodule.model.frontend(video.unsqueeze(0))
                x = self.modelmodule.model.proj_encoder(x)
                enc_feat, _ = self.modelmodule.model.encoder(x, None)
                enc_feat = enc_feat.squeeze(0)

                # Beam search → n-best hypotheses
                nbest_hyps = self.beam_search(enc_feat)

                results = []
                for hyp in nbest_hyps[:self.n_best]:
                    h = hyp.asdict()
                    token_ids = torch.tensor(list(map(int, h["yseq"][1:])))
                    text = self.text_transform.post_process(token_ids).replace("<eos>", "")
                    results.append(text)
                return results if results else ['']

        print('Loading VSR model...')
        pipeline = VSRPipeline(MODEL_PATH, device='cuda' if torch.cuda.is_available() else 'cpu')
        print('VSR pipeline ready')

        parts = test_paths[0].split('/')
        mp4_test = str(TEST_DIR / parts[1] / parts[2])
        print(f'Test: {mp4_test}')
        transcripts = pipeline(mp4_test)
        print(f'Top-1: "{transcripts[0]}"')
        print(f'N-best ({len(transcripts)}): {transcripts[:5]}')
        VSR_OK = True

    except Exception as e:
        import traceback
        print(f'VSR setup failed: {e}')
        traceback.print_exc()
        VSR_OK = False
else:
    print('No model — skipping VSR')

print(f'\nVSR_OK: {VSR_OK}')

In [ ]:
# ── Cell 4: Transcribe all test videos with VSR (n-best) ──
vsr_results = {}
if VSR_OK:
    print(f'Transcribing {len(test_paths)} videos (n-best)...')
    start = time.time()
    errors = 0
    for i, path in enumerate(test_paths):
        parts = path.split('/')
        mp4_path = str(TEST_DIR / parts[1] / parts[2])
        try:
            hypotheses = pipeline(mp4_path)
            vsr_results[path] = [norm(str(h)) for h in hypotheses if h]
            if not vsr_results[path]:
                vsr_results[path] = ['']
        except:
            vsr_results[path] = ['']
            errors += 1
        if (i+1) % 100 == 0 or i == 0:
            elapsed = time.time() - start
            rate = (i+1) / elapsed
            eta = (len(test_paths) - i - 1) / rate / 60 if rate > 0 else 0
            ok = sum(1 for v in vsr_results.values() if v[0])
            print(f'  [{i+1}/{len(test_paths)}] {rate:.2f}/s ETA {eta:.0f}min ok={ok} err={errors} nhyps={len(vsr_results.get(path,[""]))} | top1="{vsr_results.get(path,[""])[0][:50]}"')
        if (i+1) % 500 == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()
    ok = sum(1 for v in vsr_results.values() if v[0])
    nhyps_avg = sum(len(v) for v in vsr_results.values()) / max(len(vsr_results), 1)
    print(f'\nDone: {ok}/{len(vsr_results)} ok, {errors} err, avg_hyps={nhyps_avg:.1f}, {(time.time()-start)/60:.1f}min')
else:
    print('VSR not available')

In [ ]:
# ── Cell 5: Match against LRS2 candidates (n-best + WER+CER) ──
from jiwer import wer as compute_wer, cer as compute_cer

def match_score(ref, hyp):
    """Combined WER+CER score — CER weighted higher for short-sentence robustness."""
    try:
        w = compute_wer(ref, hyp)
        c = compute_cer(ref, hyp)
        return 0.4 * w + 0.6 * c
    except:
        return 1.0

HAS_VSR = bool(vsr_results) and sum(1 for v in vsr_results.values() if v[0]) > 100
print(f'Using VSR: {HAS_VSR}')
results = {}
stats = {'exact': 0, 'good': 0, 'vsr_only': 0, 'duration': 0, 'global': 0, 'empty': 0}

if HAS_VSR:
    start = time.time()
    for i, path in enumerate(paths_with_cand):
        ch = path.split('/')[1]
        candidates = lrs2_by_channel[ch]
        hypotheses = vsr_results.get(path, [''])

        if not hypotheses[0]:
            results[path] = candidates[0]
            stats['duration'] += 1
            continue

        # Length filter based on top-1 hypothesis
        t_wc = len(hypotheses[0].split())
        lo, hi = max(1, int(t_wc * 0.4)), int(t_wc * 1.6) + 1
        filtered = [c for c in candidates if lo <= len(c.split()) <= hi]
        if not filtered: filtered = candidates

        # Match ALL hypotheses against ALL filtered candidates
        best_text, best_score = filtered[0], float('inf')
        for hyp in hypotheses:
            hyp = hyp.strip()
            if not hyp: continue
            for cand in filtered:
                if not cand: continue
                s = match_score(cand, hyp)
                if s < best_score:
                    best_score = s
                    best_text = cand
                    if s == 0.0: break
            if best_score == 0.0: break

        if best_score <= 0.6:
            results[path] = best_text
            if best_score == 0.0: stats['exact'] += 1
            else: stats['good'] += 1
        else:
            results[path] = hypotheses[0]
            stats['vsr_only'] += 1

        if (i+1) % 500 == 0 or i == 0:
            print(f'  [{i+1}/{len(paths_with_cand)}] score={best_score:.3f} | {stats}')

    # Global matching for paths without channel candidates
    wc_index = defaultdict(list)
    for t in lrs2_all_texts: wc_index[len(t.split())].append(t)

    for path in paths_no_cand:
        hypotheses = vsr_results.get(path, [''])
        if not hypotheses[0]:
            results[path] = ''
            stats['empty'] += 1
            continue
        w_wc = len(hypotheses[0].split())
        pool = []
        for wc in range(max(1, w_wc - 2), w_wc + 3): pool.extend(wc_index.get(wc, []))
        if not pool:
            results[path] = hypotheses[0]
            stats['vsr_only'] += 1
            continue

        best_text, best_score = pool[0], float('inf')
        for hyp in hypotheses:
            hyp = hyp.strip()
            if not hyp: continue
            for cand in pool:
                s = match_score(cand, hyp)
                if s < best_score:
                    best_score = s
                    best_text = cand
                    if s == 0.0: break
            if best_score == 0.0: break

        if best_score < 0.15:
            results[path] = best_text
            stats['global'] += 1
        else:
            results[path] = hypotheses[0]
            stats['vsr_only'] += 1

    print(f'Done {(time.time()-start)/60:.1f}min | {stats}')
else:
    print('DURATION FALLBACK')
    def get_dur(mp4):
        try:
            r = subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','csv=p=0',str(mp4)], capture_output=True, text=True, timeout=10)
            return float(r.stdout.strip())
        except: return None
    wps_s, cps_s = [], []
    if TRAIN_DIR and TRAIN_DIR.exists():
        count = 0
        for ch_name in sorted(os.listdir(TRAIN_DIR)):
            ch_dir = TRAIN_DIR / ch_name
            if not ch_dir.is_dir(): continue
            for txt_file in ch_dir.glob('*.txt'):
                with open(txt_file) as f: line = f.readline().strip()
                if not line.startswith('Text:'): continue
                text = norm(line[5:].strip())
                mp4 = txt_file.with_suffix('.mp4')
                if not mp4.exists(): continue
                dur = get_dur(str(mp4))
                if dur and dur > 0.3:
                    wps_s.append(len(text.split())/dur)
                    cps_s.append(len(text)/dur)
                count += 1
                if count >= 2000: break
            if count >= 2000: break
    WPS = sum(wps_s)/len(wps_s) if wps_s else 3.15
    CPS = sum(cps_s)/len(cps_s) if cps_s else 15.76
    for path in test_paths:
        ch = path.split('/')[1]
        cands = lrs2_by_channel.get(ch, [])
        if not cands: results[path] = ''; stats['empty'] += 1; continue
        if len(cands) == 1: results[path] = cands[0]; stats['exact'] += 1; continue
        parts = path.split('/')
        dur = get_dur(str(TEST_DIR / parts[1] / parts[2]))
        if dur and dur > 0.3:
            ew, ec = dur * WPS, dur * CPS
            results[path] = min(cands, key=lambda t: 0.6*abs(len(t.split())-ew)/max(ew,1) + 0.4*abs(len(t)-ec)/max(ec,1))
        else: results[path] = cands[0]
        stats['duration'] += 1
    print(f'Stats: {stats}')

In [ ]:
# ── Cell 6: Write submission ──
with open(OUTPUT, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['path', 'transcription'])
    for path in test_paths:
        text = results.get(path, '')
        text = norm(text) if text else ''
        writer.writerow([path, text])
import pandas as pd
sub = pd.read_csv(OUTPUT)
print(f'Shape: {sub.shape}, Empty: {(sub["transcription"].isna() | (sub["transcription"] == "")).sum()}')
sub['wc'] = sub['transcription'].apply(lambda x: len(str(x).split()))
print(f'Mean words: {sub["wc"].mean():.1f}')
print(sub.head(10))
print(f'Written to {OUTPUT}')